> **LangChain 1.x (2026)** — built on `langchain-core==1.2.30`, `langchain==1.0.0`. See `UPDATE_2026.md`.

# Chapter 5 — Agent Evaluation & Cost Controls (v2026)

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/IvanReznikov/LangChain4LifeSciencesHealthcare/blob/main/notebooks/Chapter%2005.%20Building%20Personal%20Assistants/LC4LSH_Chapter_5_Agent_Evaluation_and_Cost_Controls.ipynb)

**Learning objectives**
- Measure tool-call validity and groundedness
- Track token usage and estimate cost per run
- Enforce reproducibility with seeds and logging
- Build a small agent eval harness

> Runtime: ~12 min (API)  
> Cost: paid LLM required  
> Data: synthetic eval cases


## Environment setup


### Secrets (Colab or local)


In [ ]:
import os
try:
    from google.colab import userdata  # type: ignore
    IN_COLAB = True
except Exception:
    userdata = None
    IN_COLAB = False
if not IN_COLAB:
    try:
        from dotenv import load_dotenv  # type: ignore
        load_dotenv()
    except Exception:
        pass

def get_secret(name, default=None):
    if IN_COLAB and userdata is not None:
        try:
            val = userdata.get(name)
            if val:
                return val
        except Exception:
            pass
    return os.getenv(name, default)

API_KEY_PROVIDER = "OPENAI"  # "GEMINI" | "OPENAI" | "GROQ" | "ANTHROPIC"
if API_KEY_PROVIDER == "OPENAI":
    os.environ["OPENAI_API_KEY"] = get_secret("LC4LSH_OPENAI_API_KEY", "sk-...")
elif API_KEY_PROVIDER == "ANTHROPIC":
    os.environ["ANTHROPIC_API_KEY"] = get_secret("LC4LSH_ANTHROPIC_API_KEY", "sk-ant-...")
elif API_KEY_PROVIDER == "GEMINI":
    os.environ["GOOGLE_API_KEY"] = get_secret("LC4LSH_GOOGLE_API_KEY", "AIza...")
elif API_KEY_PROVIDER == "GROQ":
    os.environ["GROQ_API_KEY"] = get_secret("LC4LSH_GROQ_API_KEY", "gsk_...")
print("API keys loaded for", API_KEY_PROVIDER)
os.environ["HF_TOKEN"] = get_secret("HF_TOKEN", "") or ""


### Install pinned dependencies


In [ ]:
%pip install -q "langchain==1.0.0" "langchain-core==1.2.30" "langchain-openai==1.0.0" "langchain-community==0.4.0" "langgraph>=0.2" "pydantic>=2.5" python-dotenv
# Pinned versions - Last validated: 2026-07-21 (see UPDATE_2026.md)


In [ ]:
LANGSMITH_API_KEY = get_secret("LANGSMITH_API_KEY", "lsv2_pt_...")
LANGSMITH_PROJECT = "lc4lsh-chapter5-agent-eval"
REGION = "US"
if LANGSMITH_API_KEY and LANGSMITH_API_KEY.startswith("lsv2_"):
    os.environ["LANGSMITH_TRACING"] = "true"
    os.environ["LANGSMITH_API_KEY"] = LANGSMITH_API_KEY
    os.environ["LANGSMITH_PROJECT"] = LANGSMITH_PROJECT
    print("LangSmith ON ->", LANGSMITH_PROJECT)
else:
    os.environ["LANGSMITH_TRACING"] = "false"
    print("LangSmith OFF")


## Why evaluate agents?

An agent can look right while calling the wrong tools, hallucinating evidence, or burning budget. We evaluate four things:

- **Tool-call validity** — did it call real tools with well-formed args?
- **Groundedness** — is the answer supported by retrieved evidence?
- **Reproducibility** — same input + seed → same output?
- **Cost** — tokens and dollars per run


## 1. A run record for every agent execution


In [ ]:
from pydantic import BaseModel, Field
import time

class RunRecord(BaseModel):
    question: str
    tool_calls: list[dict] = []
    answer: str = ""
    prompt_tokens: int = 0
    completion_tokens: int = 0
    latency_s: float = 0.0

    @property
    def total_tokens(self):
        return self.prompt_tokens + self.completion_tokens

    def cost_usd(self, in_per_1k=0.00015, out_per_1k=0.0006):
        return (self.prompt_tokens/1000)*in_per_1k + (self.completion_tokens/1000)*out_per_1k

print("RunRecord ready")


## 2. A tiny agent that records its run


In [ ]:
from langchain_openai import ChatOpenAI
from langchain_core.tools import tool

@tool("get_temp")
def get_temp(city: str) -> str:
    """Get a (fake) temperature for a city."""
    return f"{city}: 22C"

llm = ChatOpenAI(model="gpt-4o-mini", temperature=0)
llm_tools = llm.bind_tools([get_temp])

def run_agent(question):
    rec = RunRecord(question=question)
    t0 = time.time()
    msg = llm_tools.invoke(question)
    rec.latency_s = time.time() - t0
    usage = getattr(msg, "usage_metadata", None) or {}
    rec.prompt_tokens = usage.get("input_tokens", 0)
    rec.completion_tokens = usage.get("output_tokens", 0)
    for tc in getattr(msg, "tool_calls", []) or []:
        rec.tool_calls.append({"name": tc["name"], "args": tc["args"]})
        if tc["name"] == "get_temp":
            rec.answer = get_temp.invoke(tc["args"])
    if not rec.answer:
        rec.answer = msg.content
    return rec

rec = run_agent("What is the temperature in Basel?")
print(rec.model_dump())


## 3. Metric: tool-call validity


In [ ]:
KNOWN_TOOLS = {"get_temp"}

def tool_call_validity(rec):
    if not rec.tool_calls:
        return None
    valid = sum(1 for tc in rec.tool_calls if tc["name"] in KNOWN_TOOLS and isinstance(tc["args"], dict))
    return valid / len(rec.tool_calls)

print("Tool-call validity:", tool_call_validity(rec))


## 4. Metric: groundedness (evidence support)


In [ ]:
def groundedness(rec, evidence):
    """Fraction of answer key tokens that appear in evidence (toy proxy)."""
    ans_words = set(rec.answer.lower().split())
    ev_words = set(evidence.lower().split())
    if not ans_words:
        return 0.0
    return len(ans_words & ev_words) / len(ans_words)

print("Groundedness:", round(groundedness(rec, "Basel: 22C"), 2))


## 5. Reproducibility check


In [ ]:
def reproducible(question, n=2):
    answers = {run_agent(question).answer for _ in range(n)}
    return len(answers) == 1, answers

ok, answers = reproducible("What is the temperature in Basel?")
print("Reproducible:", ok)


## 6. Cost report


In [ ]:
print(f"prompt={rec.prompt_tokens} completion={rec.completion_tokens} total={rec.total_tokens}")
print(f"latency={rec.latency_s:.2f}s  est_cost=${rec.cost_usd():.6f}")


## Limitations & safety notes

- Groundedness here is a token-overlap proxy, not a real faithfulness judge; use an LLM-judge or RAGAS for production.
- Cost constants are illustrative; check current pricing.
- Reproducibility needs `temperature=0` and may still vary across model versions.
- **Paid API required**.


In [ ]:
# Cleanup
import gc
for _v in ("llm", "model", "agent", "graph", "app", "workflow"):
    globals().pop(_v, None)
gc.collect()
print("Cleanup complete.")


## Exercises

<details><summary>Why track tool-call validity?</summary>Malformed or hallucinated tool calls are a leading agent failure mode; measuring them catches it early.</details>

<details><summary>Why is token-overlap only a proxy for groundedness?</summary>An answer can share words yet be unsupported, or be supported with different wording; a judge model is more accurate.</details>

<details><summary>What two things drive reproducibility?</summary>Deterministic decoding (temperature=0/seed) and pinned model+library versions.</details>

### Tasks
- **Task A** - Add a `max_cost_usd` guard that raises before a run would exceed budget.
- **Task B** - Replace the proxy groundedness with a gated LLM judge that returns supported/unsupported.
- **Task C** - Log runs to JSONL and compute average cost + validity over 5 questions.
- **Task D** - Add a regression: assert `tool_call_validity == 1.0` in a test, and show a failing case.
